## A. Preparación del corpus

Cargamos el dataset arXiv Paper Abstracts (título, resumen y tópicos por paper).
Eliminamos filas con valores nulos en los campos clave y documentos duplicados
(mismo título + mismo resumen). Concatenamos título, abstract y topics en un solo
campo de texto (`text_to_embed`) para dar más contexto al modelo de embeddings y,
más adelante, al LLM generador.

In [17]:
# Instalamos las librerías necesarias para el almacenamiento y búsqueda vectorial
!pip install -q sentence-transformers chromadb pandas google-genai

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import shutil
import os

# --- A. Preparación del corpus ---

# Cargamos el dataset.
dataset_path = '/kaggle/input/datasets/spsayakpaul/arxiv-paper-abstracts/arxiv_data.csv'
df = pd.read_csv(dataset_path)

# Limpiamos filas nulas
df = df.dropna(subset=['titles', 'summaries', 'terms'])

# Eliminamos documentos completamente duplicados
df = df.drop_duplicates(
    subset=["titles", "summaries"]
).reset_index(drop=True)

print(f"Documentos después de limpiar: {len(df)}")

# Concatenamos título y abstract para generar un contexto más rico para el LLM
df["text_to_embed"] = (
    "Title: " + df["titles"] +
    "\n\nAbstract: " + df["summaries"] +
    "\n\nTopics: " + df["terms"]
)


documents = df['text_to_embed'].tolist()
metadatas = [
    {
        "title": row["titles"],
        "terms": row["terms"],
        "summary": row["summaries"]
    }
    for _, row in df.iterrows()
]
ids = [str(i) for i in range(len(df))]

Documentos después de limpiar: 38985


## C. Almacenamiento y búsqueda vectorial

Usamos ChromaDB como base de datos vectorial local, con un `PersistentClient` que
guarda el índice en disco (`/kaggle/working/arxiv_vector_db`). La colección se
configura con espacio de similitud coseno (`hnsw:space: cosine`). Antes de crear la
base se limpia cualquier índice previo para evitar segmentos HNSW huérfanos de
ejecuciones anteriores.

In [3]:
# --- C. Almacenamiento y búsqueda vectorial ---

print("Creando base de datos ChromaDB local...")

db_path = "/kaggle/working/arxiv_vector_db"

# Borramos el directorio completo para evitar
# segmentos HNSW huérfanos de ejecuciones anteriores
if os.path.exists(db_path):
    shutil.rmtree(db_path)

client = chromadb.PersistentClient(path=db_path)

collection = client.create_collection(
    name="arxiv_papers",
    metadata={"hnsw:space": "cosine"}
)

Creando base de datos ChromaDB local...


## B. Representación mediante embeddings

Usamos `all-MiniLM-L6-v2` de `sentence-transformers` para generar los vectores de
embedding de cada documento. Es un modelo ligero y rápido, adecuado para indexar
corpus grandes sin sacrificar demasiada calidad semántica. Los embeddings se
normalizan (`normalize_embeddings=True`) para poder usar similitud de coseno de
forma eficiente en la base vectorial. La generación se hace por lotes (`batch_size`)
para no saturar memoria.

In [4]:
# --- B. Representación mediante embeddings ---

print("Cargando modelo de embeddings en la GPU...")
# Utilizamos all-MiniLM-L6-v2 porque es ultraligero y rápido
model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

print("Generando vectores...")

batch_size = 1024

for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i+batch_size]

    batch_embeddings = model.encode(
        batch_docs,
        batch_size=128,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )
    collection.add(
        embeddings=batch_embeddings.tolist(),
        documents=batch_docs,
        metadatas=metadatas[i:i+batch_size],
        ids=ids[i:i+batch_size]
    )
    

print("¡Base vectorial completada!")
print(f"Total documentos: {collection.count()}")

Cargando modelo de embeddings en la GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando vectores...
¡Base vectorial completada!
Total documentos: 38985


In [4]:
print("\nProbando recuperación...\n")

query = "Applications of Graph Neural Networks"

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5
)

for i, (meta, distance) in enumerate(zip(results["metadatas"][0], results["distances"][0]),1):
    print("=" * 80)
    print(f"Resultado {i}")
    print("Título:", meta["title"])
    print("Topics:", meta["terms"])
    print(f"Distancia: {distance:.4f}")


Probando recuperación...

Resultado 1
Título: Graph Neural Networks: Taxonomy, Advances and Trends
Topics: ['cs.LG']
Distancia: 0.3157
Resultado 2
Título: A Practical Guide to Graph Neural Networks
Topics: ['cs.LG', 'cs.AI', 'cs.SI']
Distancia: 0.3537
Resultado 3
Título: Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
Topics: ['cs.LG', 'cs.NE', 'stat.ML']
Distancia: 0.3553
Resultado 4
Título: Computing Graph Neural Networks: A Survey from Algorithms to Accelerators
Topics: ['cs.LG', 'cs.DC', 'stat.ML']
Distancia: 0.3555
Resultado 5
Título: Deep Learning on Graphs: A Survey
Topics: ['cs.LG', 'cs.SI', 'stat.ML']
Distancia: 0.3611


In [5]:
def buscar(query, top_k=5):

    query_embedding = model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    resultados = []

    for meta, documento, distancia in zip(
        results["metadatas"][0],
        results["documents"][0],
        results["distances"][0]
    ):

        resultados.append({
            "title": meta["title"],
            "terms": meta["terms"],
            "summary": meta["summary"],
            "texto": documento,
            "distance": distancia
        })

    return resultados

In [6]:
# Prueba rápida de la recuperación por similitud de embeddings (antes de aplicar re-ranking),
# para verificar que la base vectorial (C) y el modelo de embeddings (B) funcionan correctamente.
docs = buscar("Applications of Graph Neural Networks")

for d in docs:
    print(f'{d["distance"]:.4f} | {d["title"]}')

0.3157 | Graph Neural Networks: Taxonomy, Advances and Trends
0.3537 | A Practical Guide to Graph Neural Networks
0.3553 | Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
0.3555 | Computing Graph Neural Networks: A Survey from Algorithms to Accelerators
0.3611 | Deep Learning on Graphs: A Survey


## D. Recuperación

Implementamos una recuperación en dos etapas: primero un top-20 rápido usando 
similitud de coseno sobre embeddings (ChromaDB), y luego un re-ranking con un 
cross-encoder (`ms-marco-MiniLM-L-6-v2`) que evalúa la relevancia query-documento 
de forma más precisa, quedándonos con el top-5 final.

In [7]:
from sentence_transformers import CrossEncoder

# --- D. Recuperación con Re-ranking ---
# Usamos un cross-encoder para reordenar los candidatos recuperados por ChromaDB
# y mejorar la precisión del top-k final (los cross-encoders comparan
# query-documento directamente, son más precisos que la similitud de embeddings sola)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def buscar_con_reranking(query, top_k_inicial=20, top_k_final=5):
    # Paso 1: recuperación inicial amplia con embeddings (rápida, sobre todo el corpus)
    candidatos = buscar(query, top_k=top_k_inicial)

    # Paso 2: reordenamos esos candidatos con el cross-encoder (más lento, pero más preciso)
    pares = [(query, c["texto"]) for c in candidatos]
    scores = reranker.predict(pares)

    for c, score in zip(candidatos, scores):
        c["rerank_score"] = float(score)

    # Ordenamos de mayor a menor score de reranking
    candidatos_ordenados = sorted(candidatos, key=lambda x: x["rerank_score"], reverse=True)

    return candidatos_ordenados[:top_k_final]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
docs_reranked = buscar_con_reranking("Applications of Graph Neural Networks")

for d in docs_reranked:
    print(f'{d["rerank_score"]:.4f} | {d["title"]}')

8.4359 | A Review of Graph Neural Networks and Their Applications in Power Systems
6.7917 | Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview
6.5711 | Graph Neural Networks: Architectures, Stability and Transferability
6.2978 | Graph Neural Networks in TensorFlow and Keras with Spektral
6.0382 | A Practical Guide to Graph Neural Networks


## E. Generación aumentada por recuperación (RAG)

Utilizamos la API de Gemini (`gemini-3.1-flash-lite`) para generar respuestas en 
lenguaje natural a partir del contexto recuperado en la etapa anterior. El prompt 
instruye al modelo a basarse únicamente en los documentos recuperados y a indicar 
explícitamente cuando no haya evidencia suficiente. Además, aplicamos un umbral sobre 
el score de re-ranking para detectar automáticamente consultas fuera de dominio.

In [9]:
from google import genai as google_genai
from kaggle_secrets import UserSecretsClient

# --- E. Generación aumentada por recuperación (RAG) ---
# La API key se obtiene de Kaggle Secrets,
# nunca queda escrita en el código ni se sube al repositorio
user_secrets = UserSecretsClient()
gemini_api_key = user_secrets.get_secret("GEMINI_API_KEY")

gemini_client = google_genai.Client(api_key=gemini_api_key)

In [10]:
def generar_respuesta_rag(query, top_k=5, umbral_relevancia=0.35):
    # 1. Recuperación con re-ranking
    evidencias = buscar_con_reranking(query, top_k_inicial=20, top_k_final=top_k)

    # 2. Verificamos si hay evidencia suficientemente relevante
    #    (los cross-encoders de ms-marco dan scores negativos/positivos;
    #    un score muy bajo indica que ningún documento es realmente relevante)
    mejor_score = max(e["rerank_score"] for e in evidencias)

    if mejor_score < umbral_relevancia:
        return {
            "respuesta": "El corpus no contiene información suficiente para responder esta consulta con confianza.",
            "evidencias": evidencias,
            "suficiente": False
        }

    # 3. Construimos el contexto a partir de las evidencias recuperadas
    contexto = "\n\n".join(
        f"[Documento {i+1}] Título: {e['title']}\nTopics: {e['terms']}\nResumen: {e['summary']}"
        for i, e in enumerate(evidencias)
    )

    prompt = f"""Eres un asistente experto en literatura científica. Responde la siguiente 
pregunta del usuario basándote ÚNICAMENTE en los documentos de contexto proporcionados.

Si el contexto no contiene información suficiente para responder, indícalo explícitamente 
en vez de inventar información.

Contexto:
{contexto}

Pregunta: {query}

Respuesta (cita los documentos relevantes usando [Documento N]):"""

    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )

    return {
        "respuesta": response.text,
        "evidencias": evidencias,
        "suficiente": True
    }

In [11]:
resultado = generar_respuesta_rag("What are the main applications of Graph Neural Networks?")
print(resultado["respuesta"])

Las aplicaciones de las Redes Neuronales de Grafos (GNN) son diversas y abarcan múltiples dominios, tal como se detalla en los documentos proporcionados:

*   **Modelado de sistemas complejos:** Se utilizan para modelar sistemas físicos, aprender huellas dactilares moleculares, predecir interfaces de proteínas y realizar tareas de clasificación de enfermedades [Documento 1].
*   **Procesamiento de datos no estructurados:** Se aplican en el razonamiento sobre estructuras extraídas de textos (árboles de dependencia) e imágenes (grafos de escenas) [Documento 1].
*   **Análisis de redes sociales e información:** Son fundamentales para estudiar grafos sociales y redes de conocimiento a gran escala, permitiendo obtener información de conjuntos de datos dinámicos que contienen miles de millones de entidades [Documento 3].
*   **Tareas de aprendizaje automático sobre grafos:** Entre las aplicaciones más comunes se incluyen la clasificación de grafos, la clasificación de nodos y la predicción d

## F. Presentación de evidencias

Para cada respuesta generada, mostramos explícitamente los documentos recuperados 
que sirvieron de contexto, junto con su score de relevancia (post re-ranking). Esto 
permite al usuario verificar la trazabilidad entre la consulta, los documentos 
recuperados y la respuesta generada por el LLM, así como detectar posibles casos de 
alucinación (información en la respuesta que no está respaldada por las evidencias).

In [12]:
def mostrar_evidencias(resultado):
    print("=" * 80)
    print("RESPUESTA GENERADA")
    print("=" * 80)
    print(resultado["respuesta"])
    print("\n" + "=" * 80)
    print("EVIDENCIAS UTILIZADAS")
    print("=" * 80)

    if not resultado["suficiente"]:
        print("(No se encontró evidencia suficientemente relevante en el corpus)")

    for i, e in enumerate(resultado["evidencias"], 1):
        print(f"\n[Documento {i}] (score de relevancia: {e['rerank_score']:.4f})")
        print(f"Título: {e['title']}")
        print(f"Topics: {e['terms']}")
        print(f"Resumen: {e['summary'][:300]}...")

In [13]:
mostrar_evidencias(resultado)


RESPUESTA GENERADA
Las aplicaciones de las Redes Neuronales de Grafos (GNN) son diversas y abarcan múltiples dominios, tal como se detalla en los documentos proporcionados:

*   **Modelado de sistemas complejos:** Se utilizan para modelar sistemas físicos, aprender huellas dactilares moleculares, predecir interfaces de proteínas y realizar tareas de clasificación de enfermedades [Documento 1].
*   **Procesamiento de datos no estructurados:** Se aplican en el razonamiento sobre estructuras extraídas de textos (árboles de dependencia) e imágenes (grafos de escenas) [Documento 1].
*   **Análisis de redes sociales e información:** Son fundamentales para estudiar grafos sociales y redes de conocimiento a gran escala, permitiendo obtener información de conjuntos de datos dinámicos que contienen miles de millones de entidades [Documento 3].
*   **Tareas de aprendizaje automático sobre grafos:** Entre las aplicaciones más comunes se incluyen la clasificación de grafos, la clasificación de nodo

In [14]:
resultado2 = generar_respuesta_rag("What is the best recipe for chocolate cake?")
mostrar_evidencias(resultado2)

RESPUESTA GENERADA
El corpus no contiene información suficiente para responder esta consulta con confianza.

EVIDENCIAS UTILIZADAS
(No se encontró evidencia suficientemente relevante en el corpus)

[Documento 1] (score de relevancia: -11.0074)
Título: Generative Adversarial Networks for Image and Video Synthesis: Algorithms and Applications
Topics: ['cs.CV']
Resumen: The generative adversarial network (GAN) framework has emerged as a powerful
tool for various image and video synthesis tasks, allowing the synthesis of
visual content in an unconditional or input-conditional manner. It has enabled
the generation of high-resolution photorealistic images and videos, ...

[Documento 2] (score de relevancia: -11.0086)
Título: High-Fidelity Synthesis with Disentangled Representation
Topics: ['cs.CV', 'eess.IV']
Resumen: Learning disentangled representation of data without supervision is an
important step towards improving the interpretability of generative models.
Despite recent advances in di

In [15]:
# Comprimimos la base vectorial para descargarla y subirla al repositorio
# de la aplicación desplegada (ver sección H)
import shutil
shutil.make_archive('/kaggle/working/arxiv_vector_db', 'zip', '/kaggle/working/arxiv_vector_db')

'/kaggle/working/arxiv_vector_db.zip'

## I. Evaluación del sistema y de la generación

Evaluamos el sistema mediante un juicio subjetivo sobre un conjunto de consultas 
representativas, considerando los siguientes criterios:

- **Corrección**: ¿la respuesta es factualmente correcta según los documentos recuperados?
- **Relevancia**: ¿la respuesta atiende directamente lo que se preguntó?
- **Fidelidad**: ¿la respuesta se basa fielmente en las evidencias, sin alucinar información no presente en ellas?
- **Integración**: ¿la respuesta combina información de varios documentos quer no de uno solo?
- **Reconocimiento de insuficiencia**: ¿el sistema detecta correctamente cuando el corpus no tiene información suficiente?

Cada consulta se califica en una escala de 1 (deficiente) a 5 (excelente) en cada criterio, 
con comentarios justificando la calificación.

In [16]:
consultas_evaluacion = [
    "What are the main applications of Graph Neural Networks?",
    "How is reinforcement learning used in robotics?",
    "Recent advances in diffusion models for image generation.",
    "Techniques for improving retrieval-augmented generation systems.",
    "What is the best recipe for chocolate cake?"  # fuera de dominio, a propósito
]

resultados_evaluacion = []

for consulta in consultas_evaluacion:
    print("=" * 80)
    print(f"CONSULTA: {consulta}")
    resultado = generar_respuesta_rag(consulta)
    mostrar_evidencias(resultado)
    resultados_evaluacion.append({"query": consulta, "resultado": resultado})
    print("\n")

CONSULTA: What are the main applications of Graph Neural Networks?
RESPUESTA GENERADA
Las Redes Neuronales de Grafos (GNN) tienen una amplia gama de aplicaciones en diversos dominios debido a su capacidad para manejar datos con relaciones complejas e interdependencias. Según los documentos proporcionados, sus principales aplicaciones incluyen:

*   **Tareas de aprendizaje automático en grafos:** Esto incluye la clasificación de grafos, la clasificación de nodos y la predicción de enlaces [Documento 3].
*   **Modelado de sistemas físicos y biológicos:** Se utilizan para modelar sistemas físicos, aprender huellas dactilares moleculares y predecir interfaces de proteínas [Documento 1].
*   **Razonamiento sobre datos no estructurados:** Se aplican al razonamiento sobre estructuras extraídas de textos (árboles de dependencia) e imágenes (grafos de escena) [Documento 1].
*   **Representación de datos irregulares:** Se emplean para modelar datos como nubes de puntos y grafos sociales [Documen

### Análisis de las consultas de evaluación

**Consulta 1 (GNN) y Consulta 2 (RL en robótica):** en ambos casos el 
sistema respondió bien. Los scores de relevancia fueron altos (4.5–7.7), las 
respuestas integraron correctamente varios documentos y pude verificar cada afirmación 
contra el [Documento N] correspondiente sin encontrar alucinaciones. Son los dos casos 
más confiables en la calidad de la respuesta.

**Consulta 3 (modelos de difusión):** el mejor documento tuvo un score alto (7.3), pero 
los siguientes bajaron notablemente (2.6–3.6). El sistema los usó igual como 
contexto, y el LLM logró distinguir cuáles eran realmente relevantes, pero esto hace 
notar una limitación: al fijar el top-5 sin importar el score, a veces se incluyen 
documentos poco relevantes solo para completar el cupo.

**Consulta 4 (RAG):** este fue el caso que más dudoso. El mejor score (1.02) fue 
mucho más bajo que en las demás consultas, y 4 de los 5 documentos tuvieron scores 
negativos. Al revisar los resultados, se entiende la razón: el corpus no tiene papers 
específicamente sobre "RAG" en el sentido actual (LLMs + recuperación), sino trabajos 
más generales de recuperación de información. El LLM respondió de forma razonable y 
fiel a esas evidencias, pero lo adecuado para respuesta correcta hubiera sido "información 
insuficiente" en vez de forzar una síntesis. El umbral de 0.35 no detectó este caso 
porque el score (1.02) técnicamente lo supera — esto genera la duda de si habría que mover 
el valor para el umbral, o si necesitaría algo más adaptativo.

**Consulta 5 (receta de pastel, fuera de dominio a propósito):** aquí sí funcionó como 
se espera. El sistema detectó que no había evidencia relevante (scores cercanos a -11) y 
respondió indicando que no tenía información suficiente, sin inventar nada.


En general, el desempeño fue bastante bueno en consultas claramente dentro o fuera del 
dominio del corpus. Donde hay más dudas es en los casos "borderline" como la consulta 
4, donde el umbral fijo no es suficiente para capturar que la evidencia es solo 
tangencialmente relevante. Se podría probar con un umbral dinámico o 
mostrar el score de confianza directamente en la interfaz, para que el usuario pueda 
juzgar por sí mismo qué tan confiable es la respuesta.

| Consulta | Corrección | Relevancia | Fidelidad | Integración de docs | Detección de insuficiencia | Comentario |
|---|---|---|---|---|---|---|
| Aplicaciones de GNN | 5 | 5 | 5 | 5 | N/A | Respuesta correcta, cada punto respaldado por una evidencia específica, cubre 5 documentos distintos sin repetir contenido. |
| RL en robótica | 5 | 5 | 5 | 5 | N/A | Integra 5 documentos con subtemas distintos (espacial, control continuo, swarm, causalidad); no hay señales de alucinación. |
| Modelos de difusión | 4 | 4 | 4 | 4 | N/A | Buena síntesis, pero incluye documentos de relevancia decreciente (restauración de imágenes es un caso límite de "generación de imágenes"); el LLM mantiene fidelidad a pesar de eso. |
| Técnicas de RAG | 3 | 3 | 4 | 3 | 2 | El LLM fue fiel a las evidencias recuperadas, pero estas evidencias mismas son solo tangencialmente relevantes al RAG moderno (limitación del corpus, no del LLM). Se esperaba que el sistema señalara baja confianza en este caso y no lo hizo — confirma lo dicho en el análisis: el umbral no capturó este caso límite. |
| Receta de pastel (fuera de dominio) | 5 | 5 | 5 | N/A | 5 | Detección correcta y explícita de que el corpus no tiene información suficiente; no se generó contenido inventado. |

* N/A = la consulta sí tenía evidencia suficiente en el corpus, por lo que este criterio no aplica.

**Promedio general:** Corrección 4.4/5, Relevancia 4.4/5, Fidelidad 4.6/5, Integración 4.25/5, 
Detección de insuficiencia 3.5/5 (basado en los 2 casos aplicables)

**Conclusión:** el sistema funciona de forma robusta en consultas alineadas con el dominio 
del corpus (ML/DL) y detecta correctamente los casos claramente fuera de dominio. La 
principal debilidad identificada es el umbral fijo de relevancia (0.35), que no distingue 
bien los casos "borderline" donde el corpus tiene información tangencialmente relacionada 
pero no realmente satisfactoria — un área de mejora futura sería usar un umbral dinámico 
o mostrar el score de confianza directamente al usuario en la interfaz.

## G. Interfaz web conversacional

La interfaz conversacional se implementó como una aplicación independiente de Streamlit 
(`app.py`), reutilizando las mismas funciones de recuperación, re-ranking y generación 
desarrolladas en este notebook. Permite ingresar consultas en lenguaje natural, muestra 
la respuesta generada junto con las evidencias utilizadas (expandibles), y mantiene un 
historial de conversación dentro de la sesión sin requerir memoria conversacional persistente.

## H. Despliegue en la nube

La base vectorial generada en la sección C (`arxiv_vector_db`) se comprime en la 
sección anterior y se descomprime dentro del repositorio de la aplicación, junto al 
`app.py`, para que `PersistentClient` pueda cargarla directamente en el entorno de 
despliegue sin necesidad de regenerarla en cada arranque.

La aplicación está desplegada en Streamlit Community Cloud y es accesible en:

**URL: https://examen-final-ri.streamlit.app/**

La API key de Gemini se gestiona mediante el sistema de Secrets de Streamlit Cloud, 
sin quedar expuesta en el código fuente ni en el repositorio.